# Execution evidence: CFO Operations Platform + AI Cost & Governance

Executed against Databricks workspace `fevm-kat-personal` (AWS us-west-2), catalog `kat_personal_catalog.cfo_febar`. Every output below was produced by running the cell; nothing was typed by hand.

## How I built it (workflow and AI tools)
- **Order:** synthetic source extracts (SAP, Salesforce, Workday, Concur, AI Gateway and billing system-table look-alikes), then Lakeflow Auto Loader ingestion into bronze, then the Unity Catalog medallion (silver/gold, metric views, a ~30-assertion plausibility safety net), then the AI cost spine and attribution, then the Genie space, then Claude insight generation with a consistency-validation harness, then the Databricks App, and finally Lakebase for operational serving and action write-back. All of it deploys as one Declarative Automation Bundle.
- **AI tools:** built with **Claude Code** as a pair programmer, using the **Databricks AI Dev Kit** skills (pipelines, Lakebase, Genie, Apps, bundles) and the Databricks CLI. A running project log carried decisions and dead ends between sessions.
- **Where AI was the force multiplier:** (1) researching the AI Gateway and billing system-table schemas and verifying the attribution mechanics (request tags, endpoint tags, budget policies) before modeling, so the synthetic tables match the real schemas; (2) generating 28 interlocking synthetic tables with engineered, discoverable business patterns plus the assertions that keep them plausible; (3) the runtime AI itself, with Claude Opus for agent and judgment calls and Haiku for composing from computed values, checked against gold-table totals; (4) this iteration, where adding the Lakeflow pipeline and the Lakebase layer, deploying, and capturing this evidence took a single Claude Code session.
- **What stayed human:** the scope (GenAI inference cost, not ML training), the personas, and the rule that every visual comes from a system table or a printed formula.


In [1]:
import os, json, time, datetime
os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "kat-personal")
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
FQ, WH = "kat_personal_catalog.cfo_febar", "f1bf4bce59e8eee2"
JOB_RUN_ID, PIPELINE_ID = 595756286301571, "e79de7c5-7300-44d2-8a8a-b9c8ecee1d0e"
APP_URL = "https://cfo-febar-7474659787238214.aws.databricksapps.com"
def q(sql):
    r = w.statement_execution.execute_statement(statement=sql, warehouse_id=WH, wait_timeout="50s")
    assert r.status.state.value == "SUCCEEDED", r.status
    return [c.name for c in r.manifest.schema.columns], (r.result.data_array or [])
def table(cols, rows):
    wd = [max(len(c), *(len(str(r[i])) for r in rows)) if rows else len(c) for i, c in enumerate(cols)]
    print("  ".join(c.ljust(wd[i]) for i, c in enumerate(cols)))
    for r in rows: print("  ".join(str(v).ljust(wd[i]) for i, v in enumerate(r)))
print("workspace:", w.config.host, "| user:", w.current_user.me().user_name, "| run at:", datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))

workspace: https://fevm-kat-personal.cloud.databricks.com | user: kateryna.savchyn@databricks.com | run at: 2026-11-02 19:13 UTC


## 1. Orchestration: the bundle job ran end to end (9/9 tasks)

In [2]:
run = w.jobs.get_run(JOB_RUN_ID)
print(f"job run {run.run_id}: {run.state.result_state.value}  ({run.run_name})")
for t in sorted(run.tasks, key=lambda t: t.start_time or 0):
    kind = "lakeflow_pipeline" if t.pipeline_task else "notebook"
    print(f"  {t.task_key:24} {kind:18} {t.state.result_state.value:8} {((t.end_time or 0)-(t.start_time or 0))/1000:6.0f}s")

job run 595756286301571: SUCCESS  (CFO Demo — Data Pipeline (cfo-febar))
  hydrate_bronze           notebook           SUCCESS     330s
  ingest_bronze            lakeflow_pipeline  SUCCESS     123s
  build_silver_gold        notebook           SUCCESS     521s
  envelope_safety_net      notebook           SUCCESS      33s
  create_metric_views      notebook           SUCCESS      12s
  provision_genie_space    notebook           SUCCESS     165s
  generate_insights        notebook           SUCCESS    1622s
  validate_consistency     notebook           SUCCESS      19s
  refresh_dashboards       notebook           SUCCESS       8s


## 2. Lakeflow: Spark Declarative Pipeline, Auto Loader into bronze streaming tables

In [3]:
evs = list(w.pipelines.list_pipeline_events(pipeline_id=PIPELINE_ID, max_results=250))
upd = sorted([e for e in evs if e.event_type == "update_progress"], key=lambda e: e.timestamp)
print("pipeline", PIPELINE_ID, "| update states:", " -> ".join(e.message.split(" is ")[-1].rstrip(".") for e in upd))
flows = {}
for e in evs:
    if e.event_type == "flow_progress" and e.message.endswith("has COMPLETED."):
        flows[e.origin.flow_name.split(".")[-1]] = True
print(f"{len(flows)} Auto Loader flows COMPLETED:", ", ".join(sorted(flows))[:1500])

pipeline e79de7c5-7300-44d2-8a8a-b9c8ecee1d0e | update states: WAITING_FOR_RESOURCES -> INITIALIZING -> RESETTING -> SETTING_UP_TABLES -> RUNNING -> COMPLETED
28 Auto Loader flows COMPLETED: bronze_ai_budget_policy, bronze_ai_gateway_external_model_spend, bronze_ai_gateway_usage, bronze_ai_list_prices, bronze_billing_usage, bronze_concur_approvals, bronze_concur_expense_items, bronze_concur_expense_reports, bronze_concur_travel_bookings, bronze_cost_center_mapping, bronze_sap_accounts_payable, bronze_sap_accounts_receivable, bronze_sap_cost_centers, bronze_sap_general_ledger, bronze_sap_profit_centers, bronze_sap_purchase_orders, bronze_sfdc_accounts, bronze_sfdc_contracts, bronze_sfdc_engagements, bronze_sfdc_forecasts, bronze_sfdc_opportunities, bronze_workday_assignments, bronze_workday_billing_rates, bronze_workday_cost_rates, bronze_workday_employees, bronze_workday_organizations, bronze_workday_positions, bronze_workday_timecards


In [4]:
cols, rows = q(f"""SELECT table_name, table_type FROM kat_personal_catalog.information_schema.tables
  WHERE table_schema='cfo_febar' AND table_name LIKE 'bronze_%' ORDER BY table_name LIMIT 6""")
table(cols, rows); print("...")
cols, rows = q(f"SELECT COUNT(*) AS bronze_streaming_tables FROM kat_personal_catalog.information_schema.tables WHERE table_schema='cfo_febar' AND table_name LIKE 'bronze_%' AND table_type='STREAMING_TABLE'")
table(cols, rows)

table_name                              table_type     
bronze_ai_budget_policy                 STREAMING_TABLE
bronze_ai_gateway_external_model_spend  STREAMING_TABLE
bronze_ai_gateway_usage                 STREAMING_TABLE
bronze_ai_list_prices                   STREAMING_TABLE
bronze_billing_usage                    STREAMING_TABLE
bronze_concur_approvals                 STREAMING_TABLE
...


bronze_streaming_tables
28                     


## 3. Unity Catalog medallion: row counts bronze to gold

In [5]:
ts = ["bronze_workday_timecards","bronze_ai_gateway_usage","silver_fact_timecards","silver_dim_projects",
      "gold_project_profitability","gold_ai_cost_observability","gold_ai_cost_by_engagement","gold_ai_cost_by_cost_center","gold_persona_insights"]
table(*q(" UNION ALL ".join(f"SELECT '{t}' AS table_name, COUNT(*) AS row_count FROM {FQ}.{t}" for t in ts)))

table_name                   row_count
bronze_workday_timecards     3219302  
bronze_ai_gateway_usage      518400   
silver_fact_timecards        3219302  
silver_dim_projects          8000     
gold_project_profitability   8000     
gold_ai_cost_observability   4825     
gold_ai_cost_by_engagement   60       
gold_ai_cost_by_cost_center  149      
gold_persona_insights        83       


## 4. Guardrails: plausibility safety net and the Gen AI consistency-validation harness

In [6]:
cols, rows = q(f"SELECT COUNT(*) AS failed_assertions FROM {FQ}.envelope_safety_net_failures"); table(cols, rows)
print("(non-strict mode: failures are logged for review, not blocking; top 3 below)")
table(*q(f"SELECT `check`, ROUND(TRY_CAST(actual AS DOUBLE),1) AS actual, expected_lo, expected_hi, unit FROM {FQ}.envelope_safety_net_failures LIMIT 3"))
print()
table(*q(f"SELECT persona, slot_type, type, LEFT(message, 95) AS message FROM {FQ}.consistency_violations"))

failed_assertions
13               
(non-strict mode: failures are logged for review, not blocking; top 3 below)


check                                       actual   expected_lo  expected_hi  unit
Annual revenue (trailing 12 months)         28.4     16           22           B   
Strict partners (Partner + Senior Partner)  4302.0   2300         2600             
Total active employees                      21301.0  25000        45000            



persona  slot_type  type       message                                                                             
finance  insight    kpi_drift  Prose says firmwide DSO = 34.3 days; canonical dashboard DSO = 22.9 days (>2% drift)
finance  insight    kpi_drift  Prose says firmwide DSO = 34.3 days; canonical dashboard DSO = 22.9 days (>2% drift)


## 5. Gen AI output: Claude-generated insights grounded in gold tables

In [7]:
cols, rows = q(f"""SELECT persona, LEFT(headline, 60) AS headline, LEFT(value, 20) AS value
  FROM {FQ}.gold_persona_insights WHERE slot_type='insight' ORDER BY persona LIMIT 6""")
table(cols, rows)

persona  headline                                   value    
admin    Accrued Billable Revenue vs Annual Budget  $23.98B  
admin    Expenses vs Forecast                       $626.89M 
admin    Project Gross Margins                      46.35%   
admin    Revenue per Partner                        $8.64M   
finance  Top Unpaid Client                          $122.53M 
finance  Days Payable Outstanding                   37.8 days


## 6. Lakebase: synced-table read + OLTP write through the app's own module

In [8]:
import sys; sys.path.insert(0, ".")
os.environ["CFO_LAKEBASE_ENDPOINT"] = "projects/cfo-ops/branches/production/endpoints/primary"
os.environ["CFO_LAKEBASE_HOST"] = w.postgres.get_endpoint(name=os.environ["CFO_LAKEBASE_ENDPOINT"]).status.hosts.host
import lakebase_store as lb
t0 = time.time(); rows = lb.read_cost_centers(limit=3)
print(f"READ {lb.SERVING_TABLE} (Lakebase synced table) in {(time.time()-t0)*1000:.0f} ms")
for r in rows: print(f"  {r[0]}  {r[1]:<11} ttm_ai_cost={float(r[3]):>10,.0f}  budget={float(r[4]):>9,.0f}  util={float(r[5]):.0f}%  {r[6]}")
aid = lb.log_action("nudge_dept_head", "cost_center", rows[0][0], amount_usd=float(rows[0][3]), actor="evidence-notebook",
                    note=f"AI budget at {float(rows[0][5]):.0f}% of annual policy")
print("WRITE cfo_ops.ai_actions ->", aid)
print("READ BACK:", json.dumps(lb.recent_actions(limit=1)[0], default=str))

READ cfo_febar.ai_budget_serving (Lakebase synced table) in 2423 ms
  CC-0022  HR          ttm_ai_cost=   769,874  budget=  100,926  util=763%  OVER_BUDGET
  CC-0036  Operations  ttm_ai_cost=   632,301  budget=  124,097  util=510%  OVER_BUDGET
  CC-0039  Executive   ttm_ai_cost=   521,122  budget=  113,792  util=458%  OVER_BUDGET


WRITE cfo_ops.ai_actions -> f60c077a-61a3-4d31-83c4-d857f692df55


READ BACK: {"action_id": "f60c077a-61a3-4d31-83c4-d857f692df55", "created_at": "2026-11-02 19:13:18.105951+00:00", "action_type": "nudge_dept_head", "target_type": "cost_center", "target_id": "CC-0022", "client_name": null, "amount_usd": "769874.00", "status": "open", "actor": "evidence-notebook", "note": "AI budget at 763% of annual policy"}


## 7. Deployed Databricks App: live API responses

In [9]:
import requests
H = w.config.authenticate()
r = requests.get(APP_URL + "/health", headers=H, timeout=60); print("GET /health", r.status_code, r.text[:160])
r = requests.get(APP_URL + "/api/ai-cost/kpis", headers=H, timeout=120); k = r.json()
print("GET /api/ai-cost/kpis", r.status_code, json.dumps({x: k[x] for x in list(k)[:8] if x != "by_dim"}, default=str)[:600])
r = requests.get(APP_URL + "/api/ai-cost/engagements", headers=H, timeout=120); e = r.json()
print("GET /api/ai-cost/engagements", r.status_code, f"{len(e)} rows; top 3:")
for x in e[:3]: print(f"  {x['client']:<14} {x['project'][:38]:<38} ai_cost={x['ai_cost']:>10,.0f}  {x['contract_type']:<16} -> {x['recoverability']}")

GET /health 200 {"app":"cfo-demo","status":"healthy","template_dir":"/app/python/source_code/templates","templates_exist":true,"version":"complete"}



GET /api/ai-cost/kpis 200 {"budgets_at_risk": 24, "frontier_pct": 26.0, "governance_coverage_pct": 89.2, "latest_month_spend": 3459591.87, "latest_pct_revenue": 0.75, "oss_pct": 74.0, "revenue_by_dim": [{"industry": "Industrials & Electronics", "location": "Atlanta", "opex": 1479835.0, "practice": "Audit", "region": "Americas", "revenue": 1868219.0}, {"industry": "Technology Media & Telecom", "location": "Chicago", "opex": 22310564.0, "practice": "Technology", "region": "Americas", "revenue": 30281002.0}, {"industry": "Public & Social Sector", "location": "New York", "opex": 13834473.0, "practice": "Accounting", "regio


GET /api/ai-cost/engagements 200 60 rows; top 3:
  Walmart        Walmart – Strategic Roadmap FY2026     ai_cost= 5,546,679  Fixed Price      -> REPRICE_AT_RENEWAL
  Bain Capital   Bain Capital – Buy-Side Due Diligence  ai_cost= 3,901,867  Retainer         -> REVIEW
  Bain Capital   Bain Capital Enterprise Strategy Refre ai_cost= 2,411,610  Time & Materials -> RECOVERABLE


## 8. Genie agent: a real question answered over the governed tables

In [10]:
Q = "Which cost centers are furthest over their AI budget, and why?"
t0 = time.time()
r = requests.post(APP_URL + "/api/chat", headers={**H, "Content-Type": "application/json"}, json={"message": Q}, timeout=400)
a = r.json()
print(f"POST /api/chat {r.status_code} in {time.time()-t0:.0f}s | status={a.get('status')}")
print("Q:", Q); print("A:", a.get("message", "")[:1600])

POST /api/chat 200 in 55s | status=success
Q: Which cost centers are furthest over their AI budget, and why?
A: # Cost Centers Furthest Over AI Budget

Across the firm, AI spend has blown through budget at a scale that is structural, not incidental. The ten worst offenders are each over by six figures, and the top driver in nearly every case is the same handful of Databricks-hosted products.

## Top 10 Cost Centers by Overage ($)

| Rank | Cost Center | Dept | Budget | Actual (TTM) | Overage $ | Overage % |
|---|---|---|---|---|---|---|
| 1 | CC-0022 | HR | $100,926 | $769,874 | $668,948 | 662.8% |
| 2 | CC-0004 | Finance | $194,986 | $831,225 | $636,239 | 326.3% |
| 3 | CC-0036 | Operations | $124,097 | $632,301 | $508,204 | 409.5% |
| 4 | CC-0009 | Executive | $163,605 | $658,588 | $494,983 | 302.5% |
| 5 | CC-0021 | IT | $172,695 | $584,536 | $411,841 | 238.5% |
| 6 | CC-0039 | Executive | $113,792 | $521,122 | $407,330 | 358.0% |
| 7 | CC-0030 | Facilities | $202,987 | $518,660 | $